# 16 — Tensor Product & Partial Trace

Put two qubits on one stage — the tensor product, written out entry by entry — then learn to ask what one of them is doing on its own: the **partial trace**, the quantum marginal, computed by hand on a $4\times4$ matrix before any helper is called. Along the way you meet the *third* way a state can be mixed: not a tossed coin (`01/13`), not decoherence (`01/15`), but being part of something larger.

**Prerequisites:** `01/05_inner_product_and_dirac` (bras and kets, $\langle k|$), `01/10_expectation_values` (the three-axis recipe, $\langle A\rangle$), `01/12_from_states_to_density_matrix` (the outer product $|\psi\rangle\langle\psi|$, $\mathrm{tr}(\rho A)$, the validity conditions), `01/13_pure_vs_mixed` (mixed states, purity, $I/2$).

**What you'll be able to do**
- Write the tensor (Kronecker) product of two qubits and of two gates entry by entry, and read the index rule $|ab\rangle \leftrightarrow k = 2a + b$.
- Build the $4\times4$ density matrix of a product state and *see why* it is $\rho_A \otimes \rho_B$.
- Define the partial trace as "replace each block by its trace", compute it by hand, and prove it is the quantum marginal — the map that reproduces every local statistic.
- Show that for a diagonal joint state the partial trace *is* the classical row sum, and exhibit a correlated (but not entangled) state whose parts are maximally mixed.

So far, one qubit at a time. Correlation, entanglement, and the inter-system structure that quantum optimal transport ultimately compares (a coupling's marginals, `04/04`) all need at least two. Today we set up that stage carefully, because everything from `01/17` to the capstone stands on it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS, CMAP_DENSITY
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS
from qot_course.quantum.gates import IDENTITY, PAULI_X, PAULI_Y, PAULI_Z, HADAMARD, apply_gate
from qot_course.quantum.density import density_matrix, purity, von_neumann_entropy, maximally_mixed
from qot_course.quantum.composite import tensor, partial_trace

viz.use_course_style()

## 1. Two systems on one stage: the tensor product by hand

One qubit is a vector with two amplitudes, $|\psi\rangle = a_0|0\rangle + a_1|1\rangle$. Two qubits, A and B, together need one amplitude for **each pair** of classical outcomes — $00, 01, 10, 11$ — so the joint state is a vector with **four** amplitudes, and the joint space is written $\mathcal H_A \otimes \mathcal H_B$, of dimension $2 \times 2 = 4$.

When A is in $|\varphi\rangle = (a_0, a_1)$ and B, independently, in $|\chi\rangle = (b_0, b_1)$, the amplitude of the pair $ab$ is the *product* of the two amplitudes, $\varphi_a\,\chi_b$ — the same rule as for independent probabilities, with amplitudes in place of probabilities. That product is the **tensor product** (for arrays: the **Kronecker product**):

$$
|\varphi\rangle \otimes |\chi\rangle
= \begin{pmatrix} a_0 \\ a_1 \end{pmatrix} \otimes \begin{pmatrix} b_0 \\ b_1 \end{pmatrix}
= \begin{pmatrix} a_0 b_0 \\ a_0 b_1 \\ a_1 b_0 \\ a_1 b_1 \end{pmatrix}
\quad\longleftrightarrow\quad
\begin{matrix} |00\rangle \\ |01\rangle \\ |10\rangle \\ |11\rangle \end{matrix}
$$

**The index rule.** The pair $ab$ sits at position $k = 2a + b$: the **left factor is the most significant digit**. $|10\rangle$ (A in $1$, B in $0$) is the third entry, $k = 2$. This is a *convention*, and it matters: the course's `tensor` and `partial_trace` read left-to-right throughout; Qiskit numbers its qubits the other way round (qubit 0 is the *rightmost* symbol), so when Qiskit enters in `01/17` its right-to-left numbering has to be translated to today's convention.

Two facts you can read off the formula:
- $\otimes$ is **bilinear**: $(\alpha|\varphi\rangle + \beta|\varphi'\rangle)\otimes|\chi\rangle = \alpha\,|\varphi\rangle\otimes|\chi\rangle + \beta\,|\varphi'\rangle\otimes|\chi\rangle$, and likewise in the right slot. The four product vectors $|ab\rangle = |a\rangle \otimes |b\rangle$ form a basis of the $4$-dimensional joint space: every two-qubit state, product or not, is a combination of them.
- The joint vector is normalised when both factors are: $\sum_{ab}|\varphi_a \chi_b|^2 = \sum_a |\varphi_a|^2 \sum_b |\chi_b|^2 = 1$.

Operators combine the same way. For $2\times2$ matrices $M$ (acting on A) and $N$ (on B), $M \otimes N$ is the $4\times4$ block matrix whose $(a, a')$ block is $M_{aa'}\,N$:

$$
M \otimes N = \begin{pmatrix} M_{00} N & M_{01} N \\ M_{10} N & M_{11} N \end{pmatrix},
\qquad
(M \otimes N)(|\varphi\rangle \otimes |\chi\rangle) = M|\varphi\rangle \otimes N|\chi\rangle,
$$

the second identity being *the* reason to define it this way: a gate on A and a gate on B, applied together, act on each factor separately. "Do nothing to B" is $M \otimes I$. Chaining two such products gives the **mixed-product rule**, $(M\otimes N)(P\otimes Q) = MP \otimes NQ$ — the same "each factor separately" identity with operators in both slots (index proof: $\sum_{c,d} M_{ac}N_{bd}P_{ca'}Q_{db'} = (MP)_{aa'}(NQ)_{bb'}$); we will lean on it in section 2 and in exercise 4.

Let's write all of this out with plain arithmetic first, and only then let `np.kron` (which `tensor` wraps) do it for us.

In [ ]:
def kron_by_hand(u, v):
    # the (2a + b)-th entry is u[a] * v[b]: left factor = most significant digit
    return np.array([u[a] * v[b] for a in range(len(u)) for b in range(len(v))], dtype=complex)

phi = apply_gate(HADAMARD, KET_0)             # A in |+>
chi = KET_0                                   # B in |0>
joint = kron_by_hand(phi, chi)

labels = ["|00>", "|01>", "|10>", "|11>"]
print("|+> (x) |0>, entry by entry:")
for k, (lab, amp) in enumerate(zip(labels, joint)):
    print(f"  k = {k} = 2*{k // 2} + {k % 2}  {lab}: {amp.real:+.4f}")
print("by hand == np.kron:  ", np.allclose(joint, np.kron(phi, chi)))
print("by hand == tensor(): ", np.allclose(joint, tensor(phi, chi)))
print("norm of the joint vector:", np.linalg.norm(joint).round(6))

# the index rule on the basis itself: |1> (x) |0> must be the third basis vector (k = 2)
print("\n|1> (x) |0> =", np.round(tensor(KET_1, KET_0).real, 3), " -> position k = 2 = 2*1 + 0")

# operators: the block formula, then the 'act on each factor separately' identity
XZ_by_hand = np.block([[PAULI_X[0, 0] * PAULI_Z, PAULI_X[0, 1] * PAULI_Z],
                       [PAULI_X[1, 0] * PAULI_Z, PAULI_X[1, 1] * PAULI_Z]])
print("\nX (x) Z, block formula == np.kron:", np.allclose(XZ_by_hand, tensor(PAULI_X, PAULI_Z)))
lhs = tensor(PAULI_X, PAULI_Z) @ tensor(phi, chi)
rhs = tensor(apply_gate(PAULI_X, phi), apply_gate(PAULI_Z, chi))
print("(X (x) Z)(|+> (x) |0>) == X|+> (x) Z|0>:", np.allclose(lhs, rhs))
print("mixed-product rule (H (x) X)(Z (x) H) == HZ (x) XH:",
      np.allclose(tensor(HADAMARD, PAULI_X) @ tensor(PAULI_Z, HADAMARD), tensor(HADAMARD @ PAULI_Z, PAULI_X @ HADAMARD)))

# bilinearity: (|0> + |1>)/sqrt2 (x) |0> = (|00> + |10>)/sqrt2
print("bilinearity:", np.allclose(tensor(phi, chi), (tensor(KET_0, KET_0) + tensor(KET_1, KET_0)) / np.sqrt(2)))

**Read the output.** The four amplitudes of $|+\rangle \otimes |0\rangle$ are $(\tfrac{1}{\sqrt2}, 0, \tfrac{1}{\sqrt2}, 0)$: A's two amplitudes $\tfrac1{\sqrt2}, \tfrac1{\sqrt2}$ each multiplied by B's $(1, 0)$, landing at $k = 0$ ($|00\rangle$) and $k = 2$ ($|10\rangle$) — B's "$0$" fixes the least significant digit. Your hand-written rule, `np.kron` and the course's `tensor` agree to the last digit; the norm is $1$; $|1\rangle \otimes |0\rangle$ is the third basis vector, as the index rule says; the block formula reproduces $X \otimes Z$; applying $X \otimes Z$ to the joint state is the same as applying $X$ to A and $Z$ to B separately; and the mixed-product rule holds on four gates. Those last two lines are the whole point of $\otimes$: independent systems, independent actions.

## 2. The joint density matrix — and why a product state is $\rho_A \otimes \rho_B$

From `01/12`, a state vector becomes a density matrix through the outer product. For the joint vector $|\Psi\rangle = |\varphi\rangle \otimes |\chi\rangle$ this is a $4\times4$ matrix, $\rho_{AB} = |\Psi\rangle\langle\Psi|$, indexed by the pairs: its entry $(ab, a'b')$ is $\Psi_{ab}\,\overline{\Psi_{a'b'}}$.

Now substitute the product rule, $\Psi_{ab} = \varphi_a \chi_b$:

$$
(\rho_{AB})_{ab,\,a'b'} = \varphi_a \chi_b\,\overline{\varphi_{a'} \chi_{b'}} = \big(\varphi_a \overline{\varphi_{a'}}\big)\big(\chi_b \overline{\chi_{b'}}\big) = (\rho_A)_{aa'}\,(\rho_B)_{bb'},
$$

which is exactly the block formula of section 1: **$\rho_{AB} = \rho_A \otimes \rho_B$**. In words: the density matrix of two independent systems is the tensor product of their density matrices, and its $(a, a')$ block is $(\rho_A)_{aa'}$ times the whole of $\rho_B$. (This is the matrix identity $(u \otimes v)(u \otimes v)^\dagger = uu^\dagger \otimes vv^\dagger$ — the mixed-product rule of section 1 with $P = u^\dagger$, $Q = v^\dagger$, using $(u\otimes v)^\dagger = u^\dagger \otimes v^\dagger$.)

For A in $|0\rangle$ and B in $|+\rangle$, $\rho_A = \mathrm{diag}(1, 0)$: the top-left block is $1 \cdot \rho_B$ and the three other blocks are $0 \cdot \rho_B$ — so the whole of $|+\rangle$'s structure, diagonal *and* coherences, must appear in the top-left $2\times2$ corner and nowhere else. Let's build it both ways and look.

In [ ]:
state_AB = tensor(KET_0, KET_PLUS)            # A in |0>, B in |+>
rho_AB = np.outer(state_AB, state_AB.conj())   # the outer product, by hand (01/12)
rho_A, rho_B = density_matrix(KET_0), density_matrix(KET_PLUS)

print("outer product == density_matrix():      ", np.allclose(rho_AB, density_matrix(state_AB)))
print("outer product == rho_A (x) rho_B:        ", np.allclose(rho_AB, tensor(rho_A, rho_B)))
print("valid density matrix (Hermitian, tr 1, PSD):",
      np.allclose(rho_AB, rho_AB.conj().T), np.isclose(np.trace(rho_AB).real, 1), np.all(np.linalg.eigvalsh(rho_AB) > -1e-12))
print("\nrho_AB (real part; all entries are real here):")
print(np.round(rho_AB.real, 3))


def show_matrices(mats, titles, basis, suptitle):
    fig, axes = plt.subplots(1, len(mats), figsize=(4.0 * len(mats), 4.2))
    for ax, m, t, lab in zip(np.atleast_1d(axes), mats, titles, basis):
        ax.imshow(m.real, cmap=CMAP_DENSITY, vmin=-1, vmax=1)
        ax.set_xticks(range(m.shape[0])); ax.set_yticks(range(m.shape[0]))
        ax.set_xticklabels(lab); ax.set_yticklabels(lab); ax.grid(False)
        for i in range(m.shape[0]):
            for j in range(m.shape[0]):
                ax.annotate(f"{m[i, j].real:.2f}", (j, i), ha="center", va="center", color=COLORS["text"], fontsize=10)
        ax.set_title(t, pad=10)
    fig.suptitle(suptitle, fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    return fig

show_matrices([rho_AB, rho_A, rho_B], [r"$\rho_{AB}$", r"$\rho_A$ = |0><0|", r"$\rho_B$ = |+><+|"],
              [labels, ["|0>", "|1>"], ["|0>", "|1>"]],
              r"A product state: $\rho_{AB} = \rho_A \otimes \rho_B$ (real parts)")
plt.show()

**Read the figure.** Left: the $4\times4$ joint density matrix, rows and columns labelled by the pairs $|00\rangle, |01\rangle, |10\rangle, |11\rangle$. All four non-zero entries, each $0.5$, sit in the top-left $2\times2$ block — and that block *is* $\rho_B = |+\rangle\langle+|$ (right panel): diagonal $\tfrac12, \tfrac12$ and coherences $\tfrac12$. The other three blocks are $0 \cdot \rho_B = 0$ because $\rho_A = \mathrm{diag}(1, 0)$ (middle panel) has a single non-zero entry at $(0,0)$. Nothing here is a coincidence of the heatmap: it is the block formula $\rho_{AB} = \rho_A \otimes \rho_B$ that your `np.allclose` line confirmed. The joint state of two independent qubits carries each qubit's information intact, in separate factors.

## 3. The partial trace by hand: replace each block by its trace

Now the question this notebook exists for. Given a joint $\rho_{AB}$ — *any* joint state, product or not — what is "the state of A alone"? The answer is the **partial trace**, and it is worth building it from the block picture rather than accepting a symbol.

Write the $4\times4$ matrix as four $2\times2$ blocks, one block $[\rho_{AB}]_{aa'}$ for each pair of A-indices (square brackets to keep "block" apart from "subsystem B"):

$$
\rho_{AB} = \begin{pmatrix} [\rho_{AB}]_{00} & [\rho_{AB}]_{01} \\ [\rho_{AB}]_{10} & [\rho_{AB}]_{11} \end{pmatrix},
\qquad \big([\rho_{AB}]_{aa'}\big)_{bb'} = (\rho_{AB})_{ab,\,a'b'} .
$$

**Tracing out B** means: keep the A-indices, and *sum over B's diagonal* — replace each block by its trace:

$$
\rho_A = \mathrm{tr}_B\,\rho_{AB}
= \begin{pmatrix} \mathrm{tr}\,[\rho_{AB}]_{00} & \mathrm{tr}\,[\rho_{AB}]_{01} \\ \mathrm{tr}\,[\rho_{AB}]_{10} & \mathrm{tr}\,[\rho_{AB}]_{11} \end{pmatrix},
\qquad
(\rho_A)_{aa'} = \sum_{b} (\rho_{AB})_{ab,\,a'b} .
$$

**Tracing out A** is the mirror image — sum over A's diagonal, i.e. add the diagonal blocks: $\rho_B = \mathrm{tr}_A\,\rho_{AB} = [\rho_{AB}]_{00} + [\rho_{AB}]_{11}$, or entry by entry $(\rho_B)_{bb'} = \sum_a (\rho_{AB})_{ab,\,ab'}$.

In bra–ket language (`01/05`), $\mathrm{tr}_B\,\rho_{AB} = \sum_b (I \otimes \langle b|)\,\rho_{AB}\,(I \otimes |b\rangle)$ — "sandwich with each basis bra of B and add up" — which is the same sum. Three properties fall straight out of the definition, and we will test each:

1. **Linear**, and **trace-preserving**: $\mathrm{tr}(\rho_A) = \sum_a\sum_b (\rho_{AB})_{ab,ab} = \mathrm{tr}(\rho_{AB}) = 1$. (It also preserves Hermiticity and positivity, so $\rho_A$ is a valid density matrix — exercise 4.)
2. **Product rule**: $\mathrm{tr}_B(M \otimes N) = M\,\mathrm{tr}(N)$ — each block of $M\otimes N$ is $M_{aa'} N$, whose trace is $M_{aa'}\,\mathrm{tr}N$. Hence for a product state $\rho_A \otimes \rho_B$ the partial trace over B returns $\rho_A$ *exactly* (and over A returns $\rho_B$).
3. **It is the quantum marginal — the only candidate.** Whatever "the state of A alone" means, it must predict every measurement made on A alone: for every observable $M$ on A, $\langle M \rangle$ computed from $\rho_A$ must equal $\langle M \otimes I\rangle$ computed from $\rho_{AB}$. The partial trace does this — two lines of index bookkeeping, using $(M\otimes I)_{a'b',\,ab} = M_{a'a}\,\delta_{b'b}$:

$$
\mathrm{tr}\big(\rho_{AB}\,(M\otimes I)\big) = \sum_{ab,\,a'b'} (\rho_{AB})_{ab,\,a'b'}\,M_{a'a}\,\delta_{b'b}
= \sum_{aa'} \Big(\sum_b (\rho_{AB})_{ab,\,a'b}\Big) M_{a'a} = \sum_{aa'} (\rho_A)_{aa'} M_{a'a} = \mathrm{tr}(\rho_A M),
$$

and it is the *unique* map that does (Nielsen & Chuang, Box 2.6: if some $\rho_A$ reproduces $\mathrm{tr}(\rho_{AB}(M\otimes I))$ for every Hermitian $M$, it is pinned down entry by entry, since the $M$'s span all $2\times2$ matrices). We test the identity numerically in section 5 on states that are not products.

Let's compute the two partial traces of section 2's matrix with plain slicing, then check against the library.

In [ ]:
def trace_out_B(rho4):
    # replace each 2x2 block by its trace: (rho_A)_{a a'} = sum_b rho_{ab, a'b}
    out = np.zeros((2, 2), dtype=complex)
    for a in range(2):
        for a_ in range(2):
            block = rho4[2 * a: 2 * a + 2, 2 * a_: 2 * a_ + 2]
            out[a, a_] = np.trace(block)
    return out

def trace_out_A(rho4):
    # add the diagonal blocks: (rho_B)_{b b'} = sum_a rho_{ab, ab'}
    return rho4[0:2, 0:2] + rho4[2:4, 2:4]

A_by_hand, B_by_hand = trace_out_B(rho_AB), trace_out_A(rho_AB)
print("tr_B by hand:\n", np.round(A_by_hand.real, 3))
print("tr_A by hand:\n", np.round(B_by_hand.real, 3))
print("\ntr_B by hand == |0><0|  :", np.allclose(A_by_hand, rho_A))
print("tr_A by hand == |+><+|  :", np.allclose(B_by_hand, rho_B))
print("by hand == partial_trace(keep=[0]):", np.allclose(A_by_hand, partial_trace(rho_AB, keep=[0], dims=[2, 2])))
print("by hand == partial_trace(keep=[1]):", np.allclose(B_by_hand, partial_trace(rho_AB, keep=[1], dims=[2, 2])))
# the same sum written with einsum: reshape (ab, a'b') -> (a, b, a', b') and contract b with b'
print("einsum 'abcb->ac' agrees:         ", np.allclose(A_by_hand, np.einsum("abcb->ac", rho_AB.reshape(2, 2, 2, 2))))

# property 1: linearity + trace preservation, on an arbitrary (non-product) Hermitian mixture
rho_1 = density_matrix(tensor(KET_PLUS, KET_1))
mix = 0.3 * rho_AB + 0.7 * rho_1
print("\nlinear:            ", np.allclose(trace_out_B(mix), 0.3 * trace_out_B(rho_AB) + 0.7 * trace_out_B(rho_1)))
print("trace preserved:   ", np.isclose(np.trace(trace_out_B(mix)).real, 1.0))
# property 2: tr_B(M (x) N) = M tr(N), with a non-normalised N to make the factor visible
M, N = HADAMARD, 2.5 * PAULI_Z + 0.7 * IDENTITY          # tr N = 1.4
print("tr_B(M (x) N) == M * tr(N):", np.allclose(trace_out_B(tensor(M, N)), M * np.trace(N)))

**Read the output.** Replacing each $2\times2$ block by its trace turns the $4\times4$ of section 2 into $\mathrm{diag}(1, 0) = |0\rangle\langle0|$: the top-left block $|+\rangle\langle+|$ has trace $1$, the three zero blocks have trace $0$. Adding the two diagonal blocks gives back $|+\rangle\langle+|$, coherences included. Both agree with the library's `partial_trace` (whose `keep=[0]` means "keep A, the left factor" — the same left-to-right convention as `tensor`) and with the one-line `einsum` that contracts B's two indices. Linearity and trace preservation hold on a mixture that is *not* a product, and $\mathrm{tr}_B(M\otimes N) = M\,\mathrm{tr}N$ holds with the factor $1.4$ showing. You now own the partial trace on paper: block, trace, done.

For a product state this is reassuring but undramatic — you get back what you put in. The interesting cases are the ones where the joint state is **not** a product. Before the quantum version (section 5), let's make sure the word "marginal" is earned literally.

## 4. The classical marginal *is* a partial trace

Take a joint probability table $p(a,b)$ for two bits — four numbers in a $2\times2$ table, non-negative, summing to $1$. Its **marginals** are the row sums $p_A(a) = \sum_b p(a,b)$ and the column sums $p_B(b) = \sum_a p(a,b)$. Two bits are **independent** when the table factorises, $p(a,b) = p_A(a)\,p_B(b)$ — i.e. the table is the Kronecker product $p_A \otimes p_B$ of its marginals, the very rule of section 1 with probabilities in place of amplitudes.

`01/12` §4 showed that a density matrix that is diagonal in the computational basis *is* a probability vector. The same holds here: place the four entries of $p(a,b)$ on the diagonal of a $4\times4$ matrix, in the order $k = 2a + b$ — $\rho_{AB} = \mathrm{diag}\big(p(0,0), p(0,1), p(1,0), p(1,1)\big)$ — and apply the block rule of section 3:

$$
\mathrm{tr}_B\,\rho_{AB} = \begin{pmatrix} p(0,0) + p(0,1) & 0 \\ 0 & p(1,0) + p(1,1) \end{pmatrix} = \mathrm{diag}\big(p_A(0), p_A(1)\big).
$$

The partial trace of a diagonal joint state is **literally** the row sum: the quantum marginal contains the classical marginal as the diagonal special case. Let's check it on an independent table and on a correlated one.

In [ ]:
pA, pB = np.array([0.3, 0.7]), np.array([0.4, 0.6])
p_indep = np.outer(pA, pB)                      # p(a,b) = pA(a) pB(b), a 2x2 table
p_corr = np.array([[0.45, 0.05],
                   [0.05, 0.45]])                # two bits that agree 90% of the time
for name, p in [("independent", p_indep), ("correlated", p_corr)]:
    rho_diag = np.diag(p.flatten())             # flatten() uses row-major order = k = 2a + b
    rowsum, colsum = p.sum(axis=1), p.sum(axis=0)
    print(f"{name} table:\n{p}")
    print("  row sums (p_A):", rowsum, "  column sums (p_B):", colsum)
    print("  tr_B(diag p) == diag(row sums):   ", np.allclose(trace_out_B(rho_diag), np.diag(rowsum)))
    print("  tr_A(diag p) == diag(column sums):", np.allclose(trace_out_A(rho_diag), np.diag(colsum)))
    print("  table == (row sums) (x) (column sums)?", np.allclose(p, np.outer(rowsum, colsum)))
    print()

**Read the output.** For both tables, tracing out B returns $\mathrm{diag}$(row sums) and tracing out A returns $\mathrm{diag}$(column sums) — the block rule and the classical sum are the same operation. The independent table is the outer product of its marginals (independence = "the table factorises" = "the diagonal $\rho_{AB}$ is a product state $\mathrm{diag}(p_A) \otimes \mathrm{diag}(p_B)$"). The correlated table is not: its marginals are both the fair coin $(0.5, 0.5)$, yet the pair is far from uniform — $90\%$ of the time the two bits agree. That is the classical seed of everything that follows: **the marginals can be perfectly random while the pair is strongly structured**, and the structure lives only in the joint table, never in its marginals. The dictionary row "marginal ↔ partial trace" is now earned in both directions.

## 5. Mixed because it is part of something larger

`01/15` ended on a promise: the partial trace is *the other way a state can be mixed*. Here it is, in its simplest form. Push the correlated table to the extreme — two bits that **always** agree, each value equally likely — and read it as a quantum state:

$$
\rho_{AB} = \tfrac12\,|00\rangle\langle00| + \tfrac12\,|11\rangle\langle11| = \mathrm{diag}(\tfrac12, 0, 0, \tfrac12).
$$

Nobody tossed a coin *on A*, and nothing decohered: this $\rho_{AB}$ is a perfectly definite description of the pair. Yet by the block rule, $\mathrm{tr}_B\,\rho_{AB} = \mathrm{diag}(\tfrac12, \tfrac12) = I/2$ — the maximally mixed state of `01/13`, purity $\tfrac12$, Bloch vector $\vec 0$. A on its own is as uncertain as a qubit can be, **because it is part of something larger** whose structure (the agreement) no local description can hold. Compare with the product state of section 2, whose marginals were pure.

Two honest remarks before you look.
- This correlated state is a *classical* correlation — a coin toss shared by two parties, a mixture of the two product states $|00\rangle$ and $|11\rangle$. It is **not** entangled (`01/18` gives the test: a mixture of product states never is). We chose it on purpose: mixed parts do **not** need entanglement. What `01/18` adds is the sharper fact that the *whole* can be **pure** — the Bell state $(|00\rangle + |11\rangle)/\sqrt2$, built in `01/17` — and the parts *still* come out $I/2$. That has no classical counterpart.
- Property 3 of section 3 says $\rho_A$ predicts every local statistic. Let's test it the way `01/10` taught: A's Bloch vector, $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$, computed from the joint state with $X\otimes I$, $Y \otimes I$, $Z\otimes I$, must equal the one read off $\rho_A$ — on the correlated coins (where every component is $0$, so the test is gentle) and on a *tilted* correlated state, $0.6\,|0\rangle\langle0|\otimes|+\rangle\langle+| + 0.4\,|1\rangle\langle1|\otimes|-\rangle\langle-|$, whose marginals lean ($\langle Z\rangle_A = 0.2$, $\langle X\rangle_B = 0.2$) and which is not a product either.

In [ ]:
rho_corr = 0.5 * density_matrix(tensor(KET_0, KET_0)) + 0.5 * density_matrix(tensor(KET_1, KET_1))
A_corr, B_corr = trace_out_B(rho_corr), trace_out_A(rho_corr)
print("tr_B of the correlated state:\n", np.round(A_corr.real, 3))
print("== I/2:", np.allclose(A_corr, maximally_mixed(2)), "  purity:", round(purity(A_corr), 3),
      "  entropy (bits):", round(von_neumann_entropy(A_corr), 3))
print("product-state marginal for comparison — purity:", round(purity(rho_A), 3))
print("joint state: purity", round(purity(rho_corr), 3), " (a 50/50 mixture of |00> and |11>, not pure)")

# Box 2.6: every local statistic of A is predicted by rho_A
print("\nlocal statistics — <M (x) I> from rho_AB  vs  <M> from rho_A:")
for name, P in [("X", PAULI_X), ("Y", PAULI_Y), ("Z", PAULI_Z)]:
    joint_avg = np.trace(rho_corr @ tensor(P, IDENTITY)).real
    local_avg = np.trace(A_corr @ P).real
    print(f"  {name}: {joint_avg:+.3f}   {local_avg:+.3f}   equal: {np.isclose(joint_avg, local_avg)}")
# a sharper test: a correlated state whose marginals are NOT I/2 (A leans to |0>, B leans to |+>)
rho_tilt = 0.6 * density_matrix(tensor(KET_0, KET_PLUS)) + 0.4 * density_matrix(tensor(KET_1, KET_MINUS))
A_tilt, B_tilt = trace_out_B(rho_tilt), trace_out_A(rho_tilt)
print("\ntilted correlated state 0.6 |0+><0+| + 0.4 |1-><1-|  (not a product:",
      not np.allclose(rho_tilt, tensor(A_tilt, B_tilt)), ")")
for name, P in [("X", PAULI_X), ("Y", PAULI_Y), ("Z", PAULI_Z)]:
    a_joint, a_loc = np.trace(rho_tilt @ tensor(P, IDENTITY)).real, np.trace(A_tilt @ P).real
    b_joint, b_loc = np.trace(rho_tilt @ tensor(IDENTITY, P)).real, np.trace(B_tilt @ P).real
    print(f"  A: <{name} (x) I> = {a_joint:+.3f}  <{name}>_A = {a_loc:+.3f}   |   B: <I (x) {name}> = {b_joint:+.3f}  <{name}>_B = {b_loc:+.3f}")
# where the structure lives: a joint statistic the marginals cannot see
print("\n<Z (x) Z> on the correlated state:", np.trace(rho_corr @ tensor(PAULI_Z, PAULI_Z)).real.round(3),
      "   <Z>_A * <Z>_B:", (np.trace(A_corr @ PAULI_Z) * np.trace(B_corr @ PAULI_Z)).real.round(3))

show_matrices([rho_corr, A_corr, B_corr],
              [r"$\rho_{AB}$ = correlated coins", r"$\rho_A = \mathrm{tr}_B\rho_{AB}$", r"$\rho_B = \mathrm{tr}_A\rho_{AB}$"],
              [labels, ["|0>", "|1>"], ["|0>", "|1>"]],
              r"A correlated (not product) state: both parts are $I/2$")
plt.show()

**Read the output and the figure.** The joint matrix (left) has weight only on $|00\rangle$ and $|11\rangle$; both marginals (middle, right) are $I/2$: purity $0.5$, one full bit of von Neumann entropy, Bloch vector $(0, 0, 0)$ — A and B are each maximally mixed, while the product state of section 2 had marginals of purity $1$. The local averages computed from the joint state with $M \otimes I$ (and $I \otimes M$ for B) match the ones read off the marginals to the last digit — zeros on the correlated coins, and the non-trivial values $\langle Z\rangle_A = 0.6 - 0.4 = 0.2$, $\langle X\rangle_B = 0.2$ on the tilted state — the partial trace *is* the local state in the only sense that matters, "predicts what you measure on A alone". And the last line shows where the structure went: $\langle Z \otimes Z\rangle = 1$ (the two $Z$-readings always agree) while $\langle Z\rangle_A\langle Z\rangle_B = 0$. The joint statistic sees the correlation; no marginal can.

So a mixed $\rho_A$ can mean three different things, and from $\rho_A$ alone you cannot tell which: a coin you tossed (`01/13`), a state that decohered (`01/15`), or a part of a larger whole — this notebook. (`01/19` will show that the second and the third are one mechanism seen from two sides: decoherence *is* becoming part of something larger — the environment — which you then trace out.) The quantum twist, that the larger whole can be **pure**, is two notebooks away.

## Your turn

1. **Swap the factors.** Build $|+\rangle \otimes |0\rangle$ by hand with `kron_by_hand`, predict from the index rule which two of the four entries are non-zero, then build its $4\times4$ $\rho$ and say — before plotting — which blocks carry the $|+\rangle$ coherences now (hint: $\rho_A = |+\rangle\langle+|$ has *four* non-zero entries). Check with `show_matrices`.
2. **The order of tracing does not matter.** Build the three-qubit product state `tensor(KET_0, KET_PLUS, KET_1)` (index $k = 4a + 2b + c$ — write the rule down first) and its $8\times8$ $\rho$. Recover $|+\rangle$ by tracing out A and C in one call, `partial_trace(rho, keep=[1], dims=[2, 2, 2])`, and in two steps (first C with `keep=[0, 1]`, then A with `keep=[1], dims=[2, 2]`). Then do it by hand on the $8\times8$: which blocks do you sum, and which traces do you take?
3. **Dial the correlation.** For $p(a,b) = \begin{pmatrix} q/2 & (1-q)/2 \\ (1-q)/2 & q/2\end{pmatrix}$ with $q \in [0, 1]$ (the two bits agree with probability $q$), embed as a diagonal $\rho_{AB}$ and plot, against $q$, the purity of $\rho_A$ and $\langle Z\otimes Z\rangle - \langle Z\rangle_A\langle Z\rangle_B$. Which one depends on $q$, which one does not, and why is that the whole message of section 5? At which $q$ is the pair independent?
4. **Harder: local gates act on the marginal.** Show by hand, using the mixed-product rule $(M\otimes N)(P\otimes Q) = MP\otimes NQ$ and $\mathrm{tr}_B(M\otimes N) = M\,\mathrm{tr}N$, that for *any* $\rho_{AB}$ and any unitary $U$ on A, $\mathrm{tr}_B\big[(U\otimes I)\,\rho_{AB}\,(U\otimes I)^\dagger\big] = U\,\rho_A\,U^\dagger$ (write $\rho_{AB}$ as a sum of products, which the basis $|a\rangle\langle a'|\otimes|b\rangle\langle b'|$ always allows). Verify numerically on `rho_corr` with $U = H$ and with $U = R_y(0.7)$ (`from qot_course.quantum.gates import ry`). Then prove that $\rho_A$ is always a valid density matrix: Hermitian (blocks $[\rho_{AB}]_{a'a} = [\rho_{AB}]_{aa'}^\dagger$), unit trace (property 1), and positive — for the last, use that $\langle\varphi|\rho_A|\varphi\rangle = \sum_b \langle\varphi, b|\rho_{AB}|\varphi, b\rangle \ge 0$. Check `is_density_matrix` (`from qot_course.quantum.density import is_density_matrix`) on the marginals of a few random two-qubit states built with `rng = np.random.default_rng(0)` (normalise a random complex 4-vector).

## What you built

- You wrote the tensor product entry by entry — $|\varphi\rangle\otimes|\chi\rangle$ has amplitudes $a_a b_b$ at index $k = 2a + b$, left factor first — and checked bilinearity, normalisation, the block formula for $M\otimes N$ and the "act on each factor separately" identity against `np.kron` and the course's `tensor`.
- You derived $\rho_{AB} = \rho_A \otimes \rho_B$ for a product state and read its block structure off the $4\times4$ matrix.
- You defined the partial trace as "replace each $2\times2$ block by its trace" (or add the diagonal blocks, for the other factor), computed it by hand, matched `partial_trace` and an `einsum`, and tested linearity, trace preservation and $\mathrm{tr}_B(M\otimes N) = M\,\mathrm{tr}N$.
- You showed the classical marginal is the diagonal case of the partial trace — row sums, column sums — on an independent and on a correlated table.
- You built a correlated-but-not-entangled joint state whose parts are $I/2$, verified that $\rho_A$ reproduces every local statistic ($\langle M\otimes I\rangle = \mathrm{tr}\,\rho_A M$), and saw the correlation live in $\langle Z\otimes Z\rangle$ where no marginal can see it — paying the promise of `01/15`: a state can be mixed because it is part of something larger.

Two systems on one stage, and the tool that asks what one of them is doing on its own, earned on paper. That is the machinery `01/17`–`01/19`, quantum mutual information (`02/10`) and the marginal constraints of a quantum coupling (`04/04`) all run on. Well built.

**Dictionary so far** (the table from `01/15`, with today's row earned — it was the first of the two forward-looking rows):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| transport plan | quantum coupling |

(The last row is still forward-looking — earned in `04/04`.)

## What's next

Section 5's correlated state was a shared coin — a classical correlation read as a quantum state. In `01/17_cnot_and_two_qubit_gates` you build the first gate that acts on *both* qubits at once, CNOT, and use it to prepare the **Bell state** $(|00\rangle + |11\rangle)/\sqrt2$ — a *pure* joint state that is not a product of anything (Qiskit enters there, with its right-to-left qubit numbering — keep today's left-to-right convention in mind when reading its matrices). Then `01/18_entanglement` takes today's partial trace of that pure state, finds $I/2$ again — a pure whole with maximally mixed parts — and that is where the correlation stops having any classical reading.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §2.1.7 (tensor products), §2.4.3 and Box 2.6 (the reduced density operator; why the partial trace is the unique map reproducing local statistics), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- M. M. Wilde, *Quantum Information Theory*, 2nd ed., §4.3 (composite quantum systems, partial trace, classically correlated states), Cambridge University Press (2017). doi:10.1017/9781316809976
- J. Watrous, *The Theory of Quantum Information*, §1.1 (the partial trace as a linear map) and §2.1 (registers and states), Cambridge University Press (2018). doi:10.1017/9781316848142

**Previous:** `notebooks/01_QuantumFoundations/15_tomography_of_a_noisy_qubit.ipynb` · **Next:** `notebooks/01_QuantumFoundations/17_cnot_and_two_qubit_gates.ipynb`